In [1]:
import os
import sys
from PIL import Image
import cv2


In [24]:
def generate_point_cloud(rgb_file, depth_file, ply_file, focal_length=35.0, scale_factor = 1000.0):
    # Load RGB image
    rgb_image = Image.open(rgb_file)
    rgb_image = rgb_image.convert("RGB")
    print(f"Loaded RGB image: {rgb_file} with size {rgb_image.size}")
    # Load depth image
    depth_image = Image.open(depth_file)
    depth_image = depth_image.convert("I")  # Convert to Intensity
    depth_image = depth_image.resize(rgb_image.size)  # Resize to match RGB image

    # Generate point cloud
    width, height  = depth_image.size
    print(f"Loaded depth image: {depth_file} with size {depth_image.size}")
    points = []

    center_x = width // 2
    center_y = height // 2  
    
    for v in range(height):
        for u in range(width):
            # print("u:", u, "v:", v, "width:", width, "height:", height  )
            color = rgb_image.getpixel((u, v))
            z = depth_image.getpixel((u, v)) / scale_factor  # Convert depth to meters
            # print("u:", u, "v:", v, "z:", z, "
            if z > 0:  # Only consider valid depth values
                x = (u - center_x) * z / focal_length  # Assuming depth is in mm, convert to meters
                y = (v - center_y) * z / focal_length   #  1000.0
                points.append((x, y, z, color[0], color[1], color[2]))

    # Save point cloud to file
    with open(ply_file, 'w') as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("property uchar red\n")
        f.write("property uchar green\n")
        f.write("property uchar blue\n")
        f.write("end_header\n")
        for point in points:
            f.write(f"{point[0]} {point[1]} {point[2]} {point[3]} {point[4]} {point[5]}\n")

In [4]:
rgb_file = "DogStar-Interiors_014-sm-optimised.jpg"
depth_file = "predicted_depth.png"
ply_file = "point_cloud.ply"

In [30]:
generate_point_cloud(rgb_file, depth_file, ply_file, focal_length=3500.0, scale_factor = 10.0)
print(f"Point cloud saved to {ply_file}")

Loaded RGB image: DogStar-Interiors_014-sm-optimised.jpg with size (3000, 2002)
Loaded depth image: predicted_depth.png with size (3000, 2002)
Point cloud saved to point_cloud.ply
